<a href="https://colab.research.google.com/github/galipaydin/pes3850/blob/main/notebooks/lab03_data_literacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 3850 · Lab 3 — From four files to one table
**Artificial Intelligence Applications in Sport · Week 3**

⏱️ About 80 minutes · 📥 This notebook downloads four small files

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce

The same four files your lecturer had on screen, turned into the single table Week 4 begins with.

Along the way you will find four faults nobody told you about, join two files that were never meant
to fit together, and decide — on purpose, and in writing — what to do about the gaps.

**One rule for the whole course:** *look at the data before you model it.* Every professional
disaster in sport analytics starts with somebody who did not.


---
## Block 1 — The four files

A season of monitoring data from one squad, as it would actually reach you: four separate files,
from four systems, that nobody designed to fit together.

| File | What it holds |
|---|---|
| `gps_sessions.csv` | one row per training session or match |
| `wellness_daily.csv` | one row per athlete per morning, filled in on a phone |
| `squad.csv` | position, date of birth, dominant limb |
| `coach_training_load.xlsx` | the fitness coach's own spreadsheet |

The data is synthetic — no real athlete is in it — but every problem in it is one you will meet.

**`rpe`** is the *rating of perceived exertion*. About thirty minutes after a session the athlete
answers one question — *how hard was that?* — on Borg's CR-10 scale, from **0** (rest) through
**5** (hard) to **10** (maximal). It measures the session from the inside: two athletes can run the
same 8 km and rate it 3 and 8. Multiplied by the session's minutes it gives **session load**, the
most widely used training-load measure in team sport. You build it in Block 6.

Every term in this notebook is explained in the course
[glossary](https://github.com/galipaydin/pes3850/blob/main/GLOSSARY.md) — keep it open in another tab.


In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/galipaydin/pes3850/main/data/week03"

def load(name, **kw):
    """Read one file from the course repository."""
    reader = pd.read_excel if name.endswith('.xlsx') else pd.read_csv
    try:
        return reader(f"{RAW}/{name}", **kw)
    except Exception as e:
        raise SystemExit(
            f"Could not download {name}.\n"
            f"  {type(e).__name__}: {e}\n\n"
            f"Open {RAW}/{name} in a browser to check it is reachable,\n"
            "or get the file from the course Drive folder and use Files → Upload\n"
            "in the panel on the left, then change load() to read it by name.")

gps      = load('gps_sessions.csv')
wellness = load('wellness_daily.csv')
squad    = load('squad.csv')

for name, d in [('gps', gps), ('wellness', wellness), ('squad', squad)]:
    print(f'{name:<9} {d.shape[0]:>6,} rows  x {d.shape[1]:>2} columns')


gps        5,090 rows  x  8 columns
wellness   4,588 rows  x  6 columns
squad         30 rows  x  4 columns


---
## Block 2 — Look at it

Four commands. You will use these four at the start of every dataset for the rest of your career.

Run them, then **read the output before scrolling on.** Something is wrong with this file in four
separate ways, and all four are visible in what prints below.


In [2]:
print('--- shape ---');            print(gps.shape)
print('\n--- columns and types ---'); print(gps.dtypes)
print('\n--- numeric summary ---');   print(gps.describe().T[['mean','min','max']].round(2))
gps.head()


--- shape ---
(5090, 8)

--- columns and types ---
athlete          object
date             object
session_type     object
duration_min    float64
distance        float64
top_speed       float64
sprint20m_s     float64
rpe              object
dtype: object

--- numeric summary ---
                 mean     min       max
duration_min    66.92   15.00    120.40
distance      5123.13  200.00  14512.10
top_speed        9.42    1.56     35.23
sprint20m_s      3.01    1.98      3.98


,athlete,date,session_type,duration_min,distance,top_speed,sprint20m_s,rpe
0,A01,2026-09-01,Recovery,52.3,3052.2,7.14,3.06,6.4
1,A02,2026-09-01,Recovery,34.6,3552.4,7.58,3.23,8.3
2,A03,2026-09-01,Recovery,41.1,3249.7,8.45,2.91,8.1
3,A04,2026-09-01,Recovery,32.6,3535.7,6.50,3.04,5.5
4,A05,2026-09-01,Recovery,38.0,2320.4,6.30,3.11,5.9


### ✏️ What did you see?

Before you read on, write down anything that looks wrong. Four things should bother you.

<details>
<summary>Only open this once you have written your list</summary>

1. **A top speed of 35.23.** No human runs at 35 m/s. Usain Bolt peaked near 12.
2. **A maximum distance of 14,512** when most sessions stop near 9,000 — and the rest of the squad
   never goes past 14,000.
3. **A 20 m sprint in 1.98 s, and another in 3.98 s.** One is impossibly fast, one impossibly slow.
4. **`rpe` is not in the numeric summary at all.** Count the rows: `describe()` shows four columns,
   but the file has five numbers in it. Look at `dtypes` to see why.

</details>


---
## Block 3 — Four faults, and what to do about each

**1 and 2 are unit problems.** Four athletes came from a club whose GPS exports speed in km/h, and
three from one that exports distance in yards. Nobody wrote this down. You find it by looking at the
maximum of every column and asking whether that number is physically possible.

**3 is two different problems wearing the same clothes.** One sprint value is a typing error; the
other is a different timing gate. Neither is an outlier in the statistical sense, and neither should
be handled by a statistical rule — see Block 3b.

**4 is a type problem.** The cell below tries to turn `rpe` into numbers the obvious way.
**It is meant to fail** — read the error it prints, because you will meet this exact
message in your own files.


In [3]:
print('How pandas stored rpe :', gps['rpe'].dtype, '  <- text, not a number')
print('The first six values  :', gps['rpe'].head(6).tolist())
print()
print('Now try to turn it into numbers, the obvious way:')
try:
    gps['rpe'].astype(float)
    print('  it worked')
except ValueError as e:
    print('  ValueError:', e)
    print()
    print('That error is the point of this cell, not a fault in the notebook.')
    print("Look at the last value above: '5,5' — written with a comma.")


How pandas stored rpe : object   <- text, not a number
The first six values  : ['6.4', '8.3', '8.1', '5.5', '5.9', '5,5']

Now try to turn it into numbers, the obvious way:
  ValueError: could not convert string to float: '5,5'

That error is the point of this cell, not a fault in the notebook.
Look at the last value above: '5,5' — written with a comma.


A Turkish keyboard, and a computer set to Turkish, writes **6,5** where an English one writes 6.5.
One comma anywhere in a column and pandas reads **the whole column** as text.

Nothing warned you. `describe()` did not complain — it simply left the column out.

### ✏️ Your task

Repair all three repairable faults. Leave the result in `gps_clean`.

* `rpe` becomes a real number, with none of the 5,090 values lost
* the four km/h athletes — `A12`, `A13`, `A14`, `A15` — have `top_speed` divided by 3.6
* the three yards athletes — `A07`, `A08`, `A09` — have `distance` divided by 1.09361

Leave the two sprint values alone for now; Block 3b is about them.

> **Careful — the athlete ids are typed two ways.** `A12` and `a12` are the same athlete. Code that
> only looks for `'A12'` misses about a quarter of that athlete's sessions, and the checkpoint will
> tell you so.


### Where the code comes from

From the slide your lecturer shows for **Block 3**. Most of the code is already typed in the cell
below; you fill in the four `___` blanks.

If you run the cell and see `NameError: name '___' is not defined`, a blank is still empty.


Run this cell first — it defines the checkpoint used below. Do not edit it.


In [4]:
def checkpoint_clean(g):
    """Checks Block 3 row by row. Says which fault is still there, and why."""
    if g is None:
        print('Nothing to check yet — put your result in `gps_clean`.'); return
    if len(g) != len(gps):
        print(f'✗ gps_clean has {len(g):,} rows; gps has {len(gps):,}. Repair values, do not drop rows.'); return
    ok = True
    num = gps['athlete'].astype(str).str.extract(r'(\d+)')[0].astype(int)
    key = 'A' + num.astype(str).str.zfill(2)
    lower = gps['athlete'].astype(str).str.startswith('a').values

    # rpe
    want_rpe = pd.to_numeric(gps['rpe'].astype(str).str.replace(',', '.'))
    if str(g['rpe'].dtype) in ('object', 'str'):
        print('✗ rpe is still text. Replace the comma with a dot, then convert to a number.'); ok = False
    elif g['rpe'].isna().sum():
        print(f"✗ rpe is a number now, but {g['rpe'].isna().sum()} values were lost on the way."); ok = False
    elif not np.allclose(g['rpe'].values, want_rpe.values):
        print('✗ rpe is a number, but some values changed. 5,5 should become 5.5.'); ok = False
    else:
        print(f"✓ rpe is a number: {len(g):,} values, none lost, mean {g['rpe'].mean():.2f}.")

    # the two unit repairs, compared on every row
    for col, who, factor, unit in [('top_speed', ['A12','A13','A14','A15'], 3.6,     'km/h'),
                                   ('distance',  ['A07','A08','A09'],       1.09361, 'yards')]:
        want = gps[col].where(~key.isin(who), gps[col] / factor)
        wrong = ~np.isclose(g[col].values, want.values)
        if not wrong.any():
            print(f'✓ {col}: every row of {", ".join(who)} converted from {unit}, nobody else touched.')
            continue
        ok = False
        n = int(wrong.sum())
        if wrong[lower].sum() == n and (key.isin(who).values & wrong).sum() == n:
            print(f"✗ {col}: {n} rows of {', '.join(who)} are still in {unit} — every one of them "
                  f"has its id typed in lower case, like '{gps['athlete'][wrong].iloc[0]}'. "
                  f"Your code matched '{key[wrong].iloc[0]}' but not '{gps['athlete'][wrong].iloc[0]}'.")
        elif (~key.isin(who).values & wrong).any():
            print(f'✗ {col}: {n} rows changed that should not have been. Only {", ".join(who)} are in {unit}.')
        else:
            print(f'✗ {col}: {n} rows of {", ".join(who)} are still in {unit}.')

    print('\nPASS — on to Block 3b.' if ok else '\nNot yet. Fix what is marked ✗.')

print('Checkpoint ready.')


Checkpoint ready.


In [5]:
# ✏️ EDIT ME — fill in the four ___ blanks. The finished code is on the Block 3 slide.

gps_clean = gps.copy()

# 1. rpe: replace the comma with a dot, then turn the text into numbers
gps_clean['rpe'] = pd.to_numeric(gps_clean['rpe'].str.replace(',', '.'))

# 2. the athlete id is typed two ways ('A07' and 'a7'): build one clean version to pick rows by
key = 'A' + gps_clean['athlete'].str.extract(r'(\d+)')[0].str.zfill(2)

# 3. km/h to m/s, for the four athletes whose GPS exported km/h
gps_clean.loc[key.isin(['A12','A13','A14','A15']), 'top_speed'] /= 3.6

# 4. yards to metres, for the three athletes recorded in yards
gps_clean.loc[key.isin(['A07','A08','A09']), 'distance'] /= 1.09361


checkpoint_clean(gps_clean)


✓ rpe is a number: 5,090 values, none lost, mean 6.31.
✓ top_speed: every row of A12, A13, A14, A15 converted from km/h, nobody else touched.
✓ distance: every row of A07, A08, A09 converted from yards, nobody else touched.

PASS — on to Block 3b.


---
## Block 3b — What an automatic outlier rule would do

The tempting move is a rule: *drop anything more than three standard deviations from the mean.*
Run it and look at what it takes.


In [ ]:
s = gps_clean['sprint20m_s'].dropna()
m, sd = s.mean(), s.std()
flagged = s[(s - m).abs() > 3 * sd]

print(f'mean {m:.3f}, sd {sd:.3f}  ->  the rule keeps {m-3*sd:.2f} to {m+3*sd:.2f}')
print(f'it removes {len(flagged)} rows:')
print(sorted(flagged.round(2).tolist()))


Two of those are the real errors. **The other eight are the fastest and slowest athletes in the
squad, timed perfectly correctly**, and the rule has just deleted them.

This is not bad luck. A 3-standard-deviation rule removes about **0.3% of any normal column**,
whether or not anything is wrong with it — so on four thousand sessions it will always throw away
about a dozen good ones. (Try the 1.5 × IQR rule instead: on this column it takes 42.)

> **The rule cannot tell an error from an athlete.** Only going back to the session, the video, or
> the person can. That is why `1.98` and `3.98` get looked at individually, and the other eight stay.

Deal with the two real errors the way a careful analyst would — mark them as unknown rather than
silently deleting the whole session, which still has valid distance and RPE in it.


In [ ]:
gps_clean = gps_clean.copy()
bad = gps_clean['sprint20m_s'].isin([1.98, 3.98])
gps_clean.loc[bad, 'sprint20m_s'] = np.nan
print(f'{bad.sum()} impossible sprint times set to NaN; the rest of those rows kept.')


---
## Block 4 — The join

`gps_clean` has the sessions. `wellness` has how the athletes felt each morning. You want them in
one table, joined on who and when.

Join it first the way almost everyone joins it the first time, and look at the three numbers.


In [ ]:
naive = gps_clean.merge(wellness, on=['athlete','date'], how='left')

print(f'rows before the join : {len(gps_clean):,}')
print(f'rows after  the join : {len(naive):,}')
print(f'wellness now missing : {naive["sleep_h"].isna().mean():.1%}')


**Read those three numbers before going on.**

The row count went **up**. A join is not supposed to invent rows, and nothing warned you that it
had. And nearly 40% of the wellness is apparently missing — but nobody stopped filling in the form.

Three things are wrong, and all three are ordinary:


In [ ]:
print('ids in gps      :', sorted(gps_clean['athlete'].unique())[:6], '...',
      gps_clean['athlete'].nunique(), 'distinct')
print('ids in squad    :', sorted(squad['athlete'].unique())[:6], '...',
      squad['athlete'].nunique(), 'distinct')
print()
dups = wellness.duplicated(subset=['athlete','date']).sum()
print(f'wellness mornings recorded twice : {dups}')
missing_ath = sorted(set(squad['athlete']) - set(wellness['athlete']))
print(f'athletes with no wellness at all : {missing_ath}')


1. **The key is typed two ways.** Whoever had the tablet wrote `A03` on some days and `a3` on
   others. Sixty distinct ids for thirty athletes. Those rows match nothing.
2. **The key is not unique on one side.** Some athletes submitted the wellness form twice in a
   morning, so one session row matches two wellness rows and becomes two rows.
3. **Three athletes were never added to the app.** They have no wellness at all — and if you reach
   for `how='inner'` to make the missing values go away, you will delete them without noticing.

### ✏️ Your task

Join them properly. Leave the result in `joined`.

* **exactly as many rows as `gps_clean`** — no more, no fewer
* missing wellness around **14%**, not 38%
* `A28`, `A29` and `A30` still present, with their wellness columns empty


### Where the code comes from

From the slide your lecturer shows for **Block 4**. Fill in the two `___` blanks.


Run this cell first — it defines the checkpoint used below. Do not edit it.


In [ ]:
def checkpoint_join(j, gps_clean, squad):
    """Checks Block 4. Diagnoses rather than grades."""
    if j is None:
        print('Nothing to check yet — put your result in `joined`.'); return
    ok = True
    if len(j) != len(gps_clean):
        d = len(j) - len(gps_clean)
        why = ('the key is not unique on one side' if d > 0
               else 'rows were dropped — check how= on your merge')
        print(f'✗ {len(j):,} rows, expected {len(gps_clean):,} ({d:+,}): {why}.'); ok = False
    else:
        print(f'✓ {len(j):,} rows, same as gps_clean. The join neither invented nor lost rows.')

    miss = j['sleep_h'].isna().mean()
    if len(j) != len(gps_clean):
        print('  (missingness cannot be judged until the row count is right)')
    elif miss > 0.25:
        print(f'✗ wellness missing {miss:.1%} — too high. Have you repaired the athlete ids?'); ok = False
    else:
        print(f'✓ wellness missing {miss:.1%}, which is the real rate.')

    gone = sorted(set(squad['athlete']) - set(j['athlete'].unique()))
    if gone:
        print(f'✗ {gone} have disappeared. An inner join drops them; use how="left".'); ok = False
    else:
        print('✓ every athlete in the squad is still in the table.')

    print('\nPASS — on to Block 5.' if ok else '\nNot yet. Fix what is marked ✗.')

print('Checkpoint ready.')


In [ ]:
# ✏️ EDIT ME — fill in the two ___ blanks. The finished code is on the Block 4 slide.

# 1. repair the id, exactly as in Block 3
n = gps_clean['athlete'].str.extract(r'(\d+)')[0]
g = gps_clean.assign(athlete='A' + n.str.zfill(2))

# 2. one wellness row per athlete per morning: which columns make a morning unique?
w = wellness.drop_duplicates(subset=___)

# 3. keep every session, matched or not
joined = g.merge(w, on=['athlete','date'], how=___)


checkpoint_join(joined, gps_clean, squad)


---
## Block 5 — Missingness is never random

Count what is missing. Then ask **when** it is missing, which is the question that matters.

There is a trap here, and it is worth meeting once. You cannot find a missing morning by looking at
the rows you have — a morning nobody filled in is not a row with a gap in it, it is **no row at
all**. To see it you first have to build the calendar of mornings that *should* exist, and then ask
which ones never arrived.


In [ ]:
w = wellness.drop_duplicates(subset=['athlete','date']).copy()
w['date'] = pd.to_datetime(w['date'])

g2 = gps_clean.copy()
g2['date'] = pd.to_datetime(g2['date'])
match_days = set(g2.loc[g2.session_type == 'Match', 'date'])

# every athlete x every morning of the season: the rows that should be there
cal = pd.MultiIndex.from_product(
        [sorted(w.athlete.unique()), pd.date_range(w.date.min(), w.date.max())],
        names=['athlete','date']).to_frame(index=False)
cal = cal.merge(w, on=['athlete','date'], how='left')
cal['after_match'] = (cal['date'] - pd.Timedelta(days=1)).isin(match_days)

rate = cal.groupby('after_match')['sleep_h'].apply(lambda s: s.isna().mean())
print(f'mornings that should exist            : {len(cal):,}')
print(f'mornings actually filled in           : {w.shape[0]:,}')
print()
print(f'wellness missing on an ordinary day   : {rate[False]:.1%}')
print(f'wellness missing after a match        : {rate[True]:.1%}')


Wellness is missing **several times more often on the day after a match** — precisely the days on
which an athlete is most fatigued and most at risk.

The missing values are not noise. They carry information about the very thing you want to predict.

* Delete those rows and you delete the hardest days of the season from your analysis.
* Fill them with the column mean and you have told the model that post-match days were ordinary.

Neither choice appears in the methods section of most papers. Both change the answer. See what each
one does here:


In [ ]:
pm = cal[cal.after_match]
print(f'mean post-match soreness, using only what was recorded : {pm.soreness.mean():.2f}')
print(f'                          gaps filled with the mean    : '
      f'{pm.soreness.fillna(cal.soreness.mean()).mean():.2f}')
print(f'                          last value carried forward   : '
      f'{cal.sort_values(["athlete","date"]).groupby("athlete").soreness.ffill()[cal.after_match.values].mean():.2f}')
print(f'ordinary days, for comparison                          : '
      f'{cal[~cal.after_match].soreness.mean():.2f}')


Filling the gaps has moved the hardest days of the season a third of the way towards being
ordinary ones. **Every gap you fill is a sentence you put in the model's mouth.**


---
## Block 6 — What is one row?

Before aggregating anything, answer this: **what is one row supposed to be?**

You have session rows. A coach plans in weeks, and an injury model usually predicts a week. So the
table has to be collapsed — but only once you know what is in it.


In [ ]:
doubles = joined.duplicated(subset=['athlete','date']).sum()
print(f'athlete-days carrying more than one session : {doubles}')
print(joined[joined.duplicated(subset=['athlete','date'], keep=False)]
      .sort_values(['athlete','date'])[['athlete','date','session_type','duration_min']]
      .head(6).to_string(index=False))


Athlete **and** date is not a unique key in your own file — there is a gym session in the afternoon.
Anyone who assumed it was unique has already been wrong today.

> **30 athletes measured 170 times each is still 30 athletes.** Week 5 is about what that does to
> the way you are allowed to split data for testing, and it starts here.


In [ ]:
j = joined.copy()
j['date'] = pd.to_datetime(j['date'])
j['week'] = ((j['date'] - j['date'].min()).dt.days // 7) + 1
j['load'] = j['rpe'] * j['duration_min']          # sRPE: the number coaches plan in

weekly = (j.groupby(['athlete','week'])
            .agg(load          = ('load','sum'),
                 distance_km   = ('distance', lambda s: s.sum()/1000),
                 sessions      = ('load','size'),
                 mean_soreness = ('soreness','mean'),
                 mean_sleep    = ('sleep_h','mean'))
            .reset_index())

print(f'{len(weekly):,} athlete-weeks from {len(j):,} sessions')
weekly.head()


`load` is **session RPE × minutes** — the most widely used training-load measure in team sport. Note
that it only exists because you repaired the comma in Block 3. **Features are built, not found.**


---
## Block 7 — The coach's own spreadsheet

The fitness coach keeps his own record, and it is beautiful to look at and nearly impossible to
analyse: one row per athlete, **one column per week**, a merged title across the top, and a totals
row dropped into the middle of the data.


In [ ]:
wide = load('coach_training_load.xlsx', sheet_name='Load', skiprows=2)
print('shape as it opens:', wide.shape)
wide.iloc[14:18, :5]


**There are 30 athletes and the file has 31 rows.** Find the extra one before you read on.

It is a `TOTAL` row sitting in the middle of the data. It disappears below only because those cells
hold formulas, which pandas reads as empty — **if somebody had pasted them as values, `TOTAL` would
now be an athlete in your dataset** with a season three times longer than anyone else's, and nothing
would have warned you.


In [ ]:
long = (wide.melt(id_vars='Athlete', var_name='week', value_name='minutes')
             .dropna(subset=['minutes']))
long['week'] = long['week'].str.lstrip('W').astype(int)

print(f'{wide.shape[0]} rows x {wide.shape[1]-1} week columns  ->  {len(long)} rows')
print(f'{long.Athlete.nunique()} athletes x {long.week.nunique()} weeks')
long.head()


One row, one observation. **Every tool in this course expects this shape** — and the shape a coach
sends you is never it.


---
## Block 8 — The table Week 4 needs

Next week you build your first model. A model does not read a season of sessions; it reads **one
table, one row per thing you are predicting, every column a number.**

Two decisions are left.

**Text columns.** `position` is 'GK', 'DF', 'MF', 'FW'. No model takes that. One column per
category, holding 0 or 1, is how it is done — and you make three columns, not four, because the
fourth is implied by the other three being zero.

**The gaps.** You already know this choice is not neutral. This notebook takes the option almost
nobody is taught: **keep the row, and record that the value was missing**, because here the
missingness is itself a fatigue signal.


In [ ]:
table = weekly.merge(squad[['athlete','position','dominant_limb']], on='athlete', how='left')
table = pd.get_dummies(table, columns=['position','dominant_limb'], drop_first=True)

table['soreness_was_missing'] = table.mean_soreness.isna().astype(int)
table['mean_soreness'] = table.mean_soreness.fillna(table.mean_soreness.mean())
table['mean_sleep']    = table.mean_sleep.fillna(table.mean_sleep.mean())

print(f'{len(table):,} rows  x  {table.shape[1]} columns')
print(f'{table.athlete.nunique()} athletes, {table.week.nunique()} weeks\n')
print(table.dtypes.to_string())
table.head()


Look at `dtypes`. Every column except `athlete` is now a number, and `athlete` is there only so you
can tell the rows apart — it is **not** a feature, and giving it to a model is a mistake you will
meet properly in Week 5.

That table is what Week 4 starts from. **You built it, from four files that did not fit together.**


---
## ✏️ EDIT ME — one athlete, one season


In [ ]:
ATHLETE = 'A07'   # ✏️ EDIT ME — any id from A01 to A30
WINDOW  = 4       # ✏️ EDIT ME — rolling mean length in weeks. Try 1, then 4, then 12.

w = weekly[weekly.athlete == ATHLETE].sort_values('week')
roll = w.load.rolling(WINDOW, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(w.week, w.load, color='#c9d6dd', label='weekly load (RPE × min)')
ax.plot(w.week, roll, color='#8A1C2B', lw=2.2, label=f'{WINDOW}-week rolling mean')
ax.set_xlabel('week of season'); ax.set_ylabel('load (AU)')
ax.set_title(f'{ATHLETE} — training load across the season')
ax.spines[['top','right']].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

print(f'{ATHLETE}: mean {w.load.mean():,.0f} AU/week, peak {w.load.max():,.0f} AU in week {int(w.loc[w.load.idxmax(),"week"])}')


---
## Block 9 — Break it 💥

Run the cell above three times, with `WINDOW` set to **1**, then **4**, then **12**.

Same athlete. Same season. Three different stories: a chaotic athlete, a periodised athlete, and an
athlete who barely varies.

**The smoothing window is an analytical choice that changes the conclusion, and it is almost never
reported.** You will meet this again in Week 9, where the acute:chronic workload ratio turns out to
rest on exactly this choice.


---
## Interpretation ✍️ **(graded)**

About 150 words:

1. **Which of the faults you found today would most distort an injury model, and why?** Not which
   was hardest to fix — which would do the most damage if it were left in.
2. **The gaps.** This notebook kept the rows and added a `soreness_was_missing` column. Name one
   situation in which that is the wrong choice.
3. **One sentence for a methods section** describing what you did to this data. If a reader could
   not reproduce your table from your sentence, it is not specific enough.


---
## AI Use Log

Complete honestly — it is part of the mark. Write *"none"* if you did not use a tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link` · 3. submit the link on ODTÜClass.

Both checkpoints must print **PASS**, and the interpretation must be written in your own words.
